# 03 — Deriving Level Thresholds from a Cost Matrix

**The problem with asserted bands.** The proposal says levels break at
0.25 / 0.50 / 0.70 / 0.85. The mid-semester report says 0.20 / 0.40 / 0.65 / 0.85.
Neither document explains where either set came from, and an examiner will ask.

**A better answer.** Write down what a mistake costs, then choose the thresholds
that minimise expected cost on held-out data. The bands become a *result* with a
derivation, and the documentation conflict dissolves.

This also produces the number the whole project rests on: **false alarms per hour,
versus a motion-only baseline.**

In [ ]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "sentrix").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

import numpy as np
import pandas as pd

In [ ]:
from sentrix.config import settings
from sentrix.fusion.features import CONTRACT_V2
from sentrix.pipeline.events import load_dataset
from sentrix.paths import EVENTLOG_DIR

print("current bands:", settings.policy.thresholds)
print("escalate dwell:", settings.policy.escalate_dwell_seconds, "s")
print("de-escalate dwell:", settings.policy.deescalate_dwell_seconds, "s")

df = load_dataset(EVENTLOG_DIR)
REAL = len(df) > 0 and "label" in df.columns and df["label"].notna().any()
print("\nreal labelled event logs:", REAL)

## The cost matrix

These are judgement calls and they belong in the report as such. What matters is
that they are **stated**, so the derived thresholds can be argued with.

A missed weapon is roughly 40x worse than a false SMS. A false *dispatch* is far
more costly than a false SMS, because it consumes emergency services — which is
the exact harm the 94-98% false-alarm statistic in your abstract describes.

In [ ]:
COSTS = {
    # level: (cost of firing when it was NOT a threat, cost of NOT firing when it WAS)
    2: (0.1,  1.0),    # UI badge + log
    3: (1.0,  5.0),    # encrypted evidence
    4: (5.0,  25.0),   # SMS + dispatch package prepared
    5: (40.0, 100.0),  # siren + voice call
}
pd.DataFrame([{"level": k, "false positive cost": v[0], "false negative cost": v[1],
               "FN:FP ratio": round(v[1] / v[0], 1)} for k, v in COSTS.items()])

In [ ]:
from sentrix.fusion.tci import TCIEngine
from sentrix.fusion.features import FeatureVector

if REAL:
    X = df[list(CONTRACT_V2)].astype(float).values
    y = (df["label"].astype(str).str.lower()
         .isin({"threat", "true", "1", "fight", "weapon", "fire", "intrusion"})).astype(int).values
    scores = np.array([TCIEngine().compute(
        FeatureVector(**dict(zip(CONTRACT_V2, row))), now=1.0).raw_tci for row in X])
else:
    print("!! synthetic demo data - illustrative only, do not report")
    rng = np.random.default_rng(7)
    n = 6000
    y = (rng.random(n) < 0.08).astype(int)          # threats are rare, as in reality
    scores = np.clip(np.where(y == 1, rng.beta(6, 2, n), rng.beta(1.4, 9, n)), 0, 1)

print(f"{len(scores):,} frames | threat prevalence {y.mean():.3f}")

In [ ]:
def optimal_threshold(scores, y, fp_cost, fn_cost):
    grid = np.linspace(0.02, 0.98, 193)
    costs = [((scores >= t) & (y == 0)).sum() * fp_cost +
             ((scores < t) & (y == 1)).sum() * fn_cost for t in grid]
    return float(grid[int(np.argmin(costs))]), grid, np.array(costs)

rows, curves = [], {}
for level, (fp, fn) in COSTS.items():
    t, grid, costs = optimal_threshold(scores, y, fp, fn)
    curves[level] = (grid, costs)
    pred = scores >= t
    tp = int((pred & (y == 1)).sum())
    fp_n = int((pred & (y == 0)).sum())
    fn_n = int((~pred & (y == 1)).sum())
    rows.append({
        "level": level,
        "derived": round(t, 3),
        "current": settings.policy.thresholds[level - 2],
        "precision": round(tp / (tp + fp_n), 3) if tp + fp_n else 0.0,
        "recall": round(tp / (tp + fn_n), 3) if tp + fn_n else 0.0,
    })
pd.DataFrame(rows)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
for level, (grid, costs) in curves.items():
    ax.plot(grid, costs / costs.max(), label=f"level {level}")
    ax.axvline(grid[int(np.argmin(costs))], ls=":", lw=1,
               color=ax.lines[-1].get_color())
ax.set_xlabel("threshold"); ax.set_ylabel("normalised expected cost")
ax.set_title("Cost-optimal thresholds" + ("" if REAL else "  [SYNTHETIC DEMO]"))
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## The headline claim: false alarms per hour

Your abstract attacks a 94-98% false-alarm rate and your approved objectives
promise a **60% or greater reduction versus a motion-only baseline**. That
comparison needs three things, and only the third is hard:

1. the SENTRIX false-alarm rate — computed below
2. the motion-only baseline rate — same footage, `tools/baseline.py`
3. **hours of continuous unattended footage from a real deployment**

Item 3 is the gap. Public CCTV datasets are trimmed and curated, so they cannot
give you a rate per hour of an *unattended room*. Record your own boring set.

Note the effect of hysteresis: alarms are counted per **incident**, not per frame.
A frame-level count inflates the number by roughly the frame rate and is not what
a user experiences.

In [ ]:
from sentrix.fusion.policy import LevelPolicy

def alarms_per_hour(scores, hz, min_level=4):
    """Count committed level transitions into >= min_level, with hysteresis."""
    policy, alarms, prev = LevelPolicy(), 0, 1
    for i, s in enumerate(scores):
        st = policy.update(float(s), now=i / hz)
        if st.level >= min_level and prev < min_level:
            alarms += 1
        prev = st.level
    hours = len(scores) / hz / 3600.0
    return alarms, hours, (alarms / hours if hours else float("nan"))

HZ = settings.pipeline.inference_fps
normal_only = scores[y == 0]

for lvl in (3, 4, 5):
    n_alarms, hours, rate = alarms_per_hour(normal_only, HZ, lvl)
    print(f"level>={lvl}: {n_alarms} false alarms over {hours:.2f} h -> {rate:.2f}/hour")

print("\nBaseline for comparison: run tools/baseline.py over the same footage.")

## Applying the derived thresholds

If the derived bands differ materially from the configured ones, record the
derivation in the report and change them in one place —
`PolicyConfig.thresholds` in `sentrix/config.py`.

Then re-run `tools/evaluate.py` and quote *those* numbers, with the commit hash
that produced them.